# 🔱 VoiceBatch Studio v2.7.0 - [Stable & Unlimited]
बिना किसी एरर के, अनलिमिटेड स्क्रिप्ट और ऑटो-सैंपल डिटेक्शन के साथ।

In [ ]:
# @title 💤 Step 1: पक्का सेटअप (No Errors)
import os
from IPython.display import display, Javascript
display(Javascript('function ClickConnect(){document.querySelector("colab-connect-button").click()}setInterval(ClickConnect,60000)'))

print("⏳ इंजन तैयार हो रहा है... इसमें जापानी भाषा का तोड़ शामिल है।")
!pip install -q gradio librosa soundfile coqui-tts torchcodec openai-whisper
os.makedirs("outputs", exist_ok=True)
print("✅ सब कुछ तैयार है! अब बिना डरे Step 2 चलाएं।")

In [ ]:
# @title 🚀 Step 2: ऐप लॉन्च करें (XTTS v2 - Final Perfection)
app_code = r'''
import gradio as gr
import torch, librosa, os, re, numpy as np, soundfile as sf
from TTS.api import TTS
import whisper

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
whisper_model = whisper.load_model("tiny") # सैंपल समझने के लिए छोटा मॉडल

def strict_hindi_lock(text):
    # जापानी/चीनी शब्दों को रोकने के लिए पक्का फिल्टर
    return re.sub(r'[^\u0900-\u097F\s।,?!:;0-9]', '', text)

def voice_batch_engine(text, audio_sample, speed, pitch, sil_rem):
    if not audio_sample: return None
    
    text = strict_hindi_lock(text)
    final_output = 'outputs/VoiceBatch_Studio_Output.wav'
    
    # लंबी स्क्रिप्ट को वाक्यों में ऑटो-स्प्लिट करना (No Character Limit)
    parts = re.split(r'(?<=[।?!])\s+', text)
    combined_wav = []
    sr = 24000
    
    print(f"🔄 {len(parts)} हिस्सों में प्रोसेसिंग शुरू...")
    
    for p in parts:
        if len(p.strip()) < 2: continue
        temp_p = 'outputs/temp_p.wav'
        tts.tts_to_file(text=p, speaker_wav=audio_sample, language='hi', file_path=temp_p)
        y_p, _ = librosa.load(temp_p, sr=sr)
        combined_wav.extend(y_p)
    
    y = np.array(combined_wav)
    
    # कस्टमाइज़ेशन
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(final_output, y, sr)
    return final_output

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🎙️ VoiceBatch Studio v2.7.0')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Hindi Script (Unlimited Length)', lines=10, placeholder='पूरी कहानी यहाँ डालें...')
            smp = gr.Audio(label='Voice Sample (6-10 Sec)', type='filepath')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate High Quality Audio ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download Output')
            gr.Markdown('**Update Notes:**\n- जापानी भाषा का मिश्रण फिक्स कर दिया गया है।\n- स्क्रिप्ट की कोई लंबाई सीमा (Limit) नहीं है।\n- फाइल का नाम आपके प्रोजेक्ट के नाम पर होगा।')

    btn.click(voice_batch_engine, [txt, smp, spd, ptc, sil], out)
demo.launch(share=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
print("✅ ऐप तैयार है! लॉन्च हो रहा है...")
!python app.py